# Late Delivery Prediction
A leakage-conscious binary-classification workflow using the real Olist PostgreSQL database.

## 1. Business Problem
Flag orders at elevated risk of late delivery early enough for operational intervention. Because late orders are uncommon, accuracy alone would reward a model that rarely flags risk.

## 2. Target Definition
The unit is one delivered order. `is_late_delivery=1` when actual customer delivery is later than the estimated delivery timestamp. Actual delivery is used only to construct the training label.

## 3. Prediction-Time Assumptions
Prediction occurs at or shortly after purchase. Calendar, customer region, basket, product catalog, seller location, checkout payment, freight quote, and estimated delivery window are available.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.late_delivery.build_dataset import build_modeling_dataset
from src.models.late_delivery.features import audit_features
from src.models.late_delivery.train import main as train_model
from src.models.late_delivery.predict import predict_late_delivery

INFO: generated new fontManager


## 4. Leakage Audit
Actual delivery/carrier timestamps, delivery duration and delay, reviews, target, and high-cardinality identifiers are prohibited. The audit fails before training if an undocumented or prohibited feature enters the matrix.

In [2]:
dataset = build_modeling_dataset()
display(audit_features(dataset))

,feature,available_at_prediction_time,rationale
0,purchase_month,True,Known from purchase timestamp.
1,purchase_day_of_week,True,Known from purchase timestamp.
2,purchase_hour,True,Known from purchase timestamp.
3,customer_zip_region,True,Coarse region derived from the known customer ...
4,item_value,True,Sum of ordered item prices known at purchase.
5,freight_value,True,Quoted order-item freight known at purchase.
6,item_count,True,Basket composition known at purchase.
7,unique_products,True,Basket composition known at purchase.
8,seller_count,True,Sellers fulfilling the basket are known at pur...
9,total_product_weight_g,True,Catalog attributes known before fulfillment.


## 5. Modeling Dataset
Items, products, sellers, and payments are aggregated to order grain in PostgreSQL before joining. Uniqueness is asserted in code.

In [3]:
print('Shape:', dataset.shape)
print('Duplicate order IDs:', dataset.order_id.duplicated().sum())
display(dataset[['purchase_timestamp','is_late_delivery']].describe(include='all'))

Shape: (96470, 24)
Duplicate order IDs: 0


,purchase_timestamp,is_late_delivery
count,96470,96470.000000
mean,2018-01-01 23:17:43.624412,0.081124
min,2016-09-15 12:16:38,0.000000
25%,2017-09-14 08:56:46.500000,0.000000
50%,2018-01-20 19:34:43.500000,0.000000
75%,2018-05-05 18:29:50.250000,0.000000
max,2018-08-29 15:00:37,1.000000
std,NaN,0.273026


## 6. Temporal Split
The earliest 70% trains models, the next 15% supports model and threshold selection, and the latest 15% remains untouched until final evaluation.

## 7. Preprocessing
Train-fitted `ColumnTransformer` pipelines median-impute numeric inputs, mode-impute categorical inputs, and one-hot encode categories. Logistic regression additionally scales numeric features.

## 8. Baseline
A prior-probability `DummyClassifier` establishes the naive PR-AUC baseline. Logistic regression provides an interpretable weighted baseline.

## 9. Model Comparison
Logistic regression, random forest, and histogram gradient boosting are compared by validation PR-AUC with modest fixed configurations. The following cell executes the complete reproducible workflow.

In [4]:
train_model()

INFO: Dataset rows=96,470 late=7,826 prevalence=8.112%


INFO: 
     split  rows               start                 end  late_orders  late_prevalence
     train 67529 2016-09-15 12:16:38 2018-04-15 20:12:35         6096         0.090272
validation 14470 2018-04-15 20:17:11 2018-06-21 08:29:29          773         0.053421
      test 14471 2018-06-21 08:41:07 2018-08-29 15:00:37          957         0.066132


INFO: Training DummyClassifier


INFO: Training LogisticRegression


INFO: Training RandomForestClassifier


INFO: Training HistGradientBoostingClassifier


{
  "model_name": "LogisticRegression",
  "features": [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "customer_zip_region",
    "item_value",
    "freight_value",
    "item_count",
    "unique_products",
    "seller_count",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "average_product_width_cm",
    "payment_value",
    "payment_installments",
    "estimated_delivery_window_days",
    "customer_state",
    "dominant_product_category",
    "dominant_seller_state",
    "same_customer_seller_state",
    "payment_type"
  ],
  "threshold": 0.55,
  "created_at_utc": "2026-08-22T14:46:39.505239+00:00",
  "training_date_range": {
    "start": "2016-09-15T12:16:38",
    "end": "2018-04-15T20:12:35"
  },
  "validation_date_range": {
    "start": "2018-04-15T20:17:11",
    "end": "2018-06-21T08:29:29"
  },
  "test_date_range": {
    "start": "2018-06-21T08:41:07",
    "end": "2018-08-29T15:00:37"
  },
  "dataset_

In [5]:
report_dir = PROJECT_ROOT / 'reports' / 'modeling'
metrics = pd.read_csv(report_dir / 'model_metrics.csv')
display(metrics[['model','split','pr_auc','roc_auc','precision','recall','f1','balanced_accuracy']])

,model,split,pr_auc,roc_auc,precision,recall,f1,balanced_accuracy
0,DummyClassifier,train,0.090272,0.500000,0.000000,0.000000,0.000000,0.500000
1,DummyClassifier,validation,0.053421,0.500000,0.000000,0.000000,0.000000,0.500000
2,LogisticRegression,train,0.215253,0.712834,0.162179,0.640256,0.258802,0.656022
3,LogisticRegression,validation,0.157964,0.767819,0.114109,0.716688,0.196873,0.701339
4,RandomForestClassifier,train,0.494246,0.903035,0.345401,0.743438,0.471666,0.801814
5,RandomForestClassifier,validation,0.140506,0.721453,0.175812,0.203105,0.188475,0.574685
6,HistGradientBoostingClassifier,train,0.354107,0.837673,0.238992,0.751476,0.362650,0.757015
7,HistGradientBoostingClassifier,validation,0.127637,0.711535,0.148760,0.279431,0.194157,0.594596


## 10. Threshold Selection
Thresholds from 0.05–0.80 are evaluated only on validation data. F2 emphasizes recall because false negatives are late orders that receive no intervention.

In [6]:
thresholds = pd.read_csv(report_dir / 'threshold_metrics.csv')
display(thresholds.sort_values('f2', ascending=False).head(10))

,threshold,precision,recall,f1,f2
50,0.55,0.135172,0.633894,0.222829,0.364746
51,0.56,0.139349,0.609314,0.226824,0.363875
49,0.54,0.129863,0.652005,0.216588,0.361394
52,0.57,0.141151,0.580854,0.227112,0.357883
48,0.53,0.125091,0.664942,0.210569,0.356895
47,0.52,0.122097,0.686934,0.207341,0.356807
46,0.51,0.118944,0.705045,0.203548,0.355095
54,0.59,0.147617,0.532988,0.231201,0.350161
53,0.58,0.142283,0.549806,0.226064,0.349564
55,0.60,0.152199,0.514877,0.234947,0.348695


## 11. Final Test Evaluation
Only the selected pipeline and validation-selected threshold are applied to the chronological final test period.

In [7]:
metadata = json.loads((PROJECT_ROOT / 'models' / 'late_delivery_metadata.json').read_text())
display(metadata['test_metrics'])

{'pr_auc': 0.11316110412202061,
 'roc_auc': 0.6574285206610304,
 'precision': 0.09962406015037593,
 'recall': 0.664576802507837,
 'f1': 0.17327339599509603,
 'balanced_accuracy': 0.6196200573143003,
 'tn': 7766,
 'fp': 5748,
 'fn': 321,
 'tp': 636}

## 12. Model Interpretation
Permutation importance reports validation PR-AUC change for raw features. Importance is predictive association, not causation.

In [8]:
display(pd.read_csv(report_dir / 'feature_importance.csv').head(15))

,feature,importance_mean,importance_std
0,estimated_delivery_window_days,0.075397,0.002364
1,customer_state,0.055225,0.006603
2,same_customer_seller_state,0.046155,0.004014
3,customer_zip_region,0.042588,0.002237
4,dominant_seller_state,0.015778,0.002520
5,payment_value,0.004994,0.000695
6,item_count,0.002553,0.000884
7,seller_count,0.002005,0.000988
8,freight_value,0.001765,0.001421
9,average_product_width_cm,0.001757,0.000433


## 13. Error Analysis
False negatives, false positives, state, order-value band, major category, and time segments are exported for review. Small segments require cautious interpretation.

In [9]:
display(pd.read_csv(report_dir / 'error_analysis_state.csv').sort_values('late_orders', ascending=False).head(12))
display(pd.read_csv(report_dir / 'error_analysis_examples.csv').head())

,customer_state,orders,late_orders,late_rate,true_positives,false_negatives,false_positives,recall,false_negative_rate
25,SP,6750,643,0.095259,383,260,1524,0.595645,0.404355
18,RJ,1625,88,0.054154,86,2,1051,0.977273,0.022727
10,MG,1581,33,0.020873,19,14,510,0.575758,0.424242
17,PR,743,24,0.032301,7,17,187,0.291667,0.708333
22,RS,720,24,0.033333,14,10,382,0.583333,0.416667
4,BA,458,22,0.048035,19,3,361,0.863636,0.136364
23,SC,462,16,0.034632,11,5,318,0.687500,0.312500
11,MS,96,15,0.156250,15,0,62,1.000000,0.000000
6,DF,356,14,0.039326,9,5,180,0.642857,0.357143
7,ES,272,13,0.047794,13,0,184,1.000000,0.000000


,order_id,purchase_timestamp,customer_state,dominant_product_category,item_value,actual,predicted,error_type
0,374289395a111a64a05644b1f59ef07d,2018-06-21 08:45:30,DF,health_beauty,122.99,0,1,false_positive
1,ead4584d779958e7605fe628d3f253f0,2018-06-21 09:08:19,CE,watches_gifts,155.00,0,1,false_positive
2,a77da241d61b2939dba10220b807d732,2018-06-21 09:08:51,CE,construction_tools_construction,498.00,0,1,false_positive
3,19b99fd79d81fab77d8b7ba4f7a5c799,2018-06-21 09:31:04,MA,health_beauty,66.99,0,1,false_positive
4,b552bf4c093b6c4804fa08eaee94a786,2018-06-21 09:36:30,BA,housewares,17.90,1,0,false_negative


## 14. Business Recommendations
Use predicted risk for an operational review queue, prioritize false-negative cost in threshold policy, and pilot interventions before claiming business impact. Monitor precision, recall, capacity, geography, and drift.

In [10]:
display(predict_late_delivery(dataset.tail(5)))

,late_delivery_probability,predicted_late_delivery,risk_label
96465,0.572569,1,HIGH
96466,0.539599,0,MEDIUM
96467,0.277756,0,MEDIUM
96468,0.520794,0,MEDIUM
96469,0.583591,1,HIGH


## 15. Limitations and Next Steps
Add live carrier capacity, weather, holidays, seller workload, and route features; calibrate probabilities; use rolling temporal validation; define intervention costs; and monitor drift. No causal conclusion follows from feature importance.